# Tái hiện V-SQL và hai cải tiến trên Google Colab

Notebook này chạy lại thực nghiệm của bài báo **V-SQL: A View-based Two-stage Text-to-SQL Framework** (arXiv:2502.15686)
trên bộ **BIRD mini-dev** (500 câu, 11 CSDL), rồi đánh giá **hai cải tiến** của nhóm:

| Ký hiệu | Cấu hình | Nội dung |
|---|---|---|
| `base` | đối chứng | Sinh SQL một lần trên toàn bộ bảng gốc |
| `vsql` | **tái hiện bài báo** | View theo phép JOIN của SQL đáp án + LLM tái tạo SQL về bảng gốc |
| `*_fix` | **Cải tiến 1 (CT5)** | Sửa lỗi theo lược đồ: chạy SQL, đọc lỗi SQLite, sửa tên/alias, tự thêm JOIN theo khóa ngoại. Không gọi thêm LLM |
| `mrv` | **Cải tiến 2 (CT7)** | Bỏ phiếu theo kết quả thực thi giữa 4 cách biểu diễn lược đồ (bảng gốc, view tự động, view + bảng, truy xuất) |

**Cách dùng:** chạy lần lượt từng ô từ trên xuống (`Ctrl+F9` để chạy tất cả). Hướng dẫn chi tiết:
`docs/HUONG_DAN_COLAB.md` trong repo https://github.com/chloenguyen167/vsqlplus.

**Thời gian (ước tính):** khoảng 6 lượt gọi LLM cho mỗi câu, tức ~3.000 lượt cho 500 câu. Với GPU T4 miễn phí và model
8–9B, mất khoảng 2–4 giờ; ô chạy chính in ra thời gian còn lại thực tế. Nếu Colab bị ngắt, chạy lại các ô: chương trình
tự bỏ qua phần đã xong (kết quả được lưu vào Google Drive).

## Bước 0. Chọn GPU
Menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU** (bản miễn phí) rồi **Save**.
Nếu có Colab Pro, chọn **L4** hoặc **A100** để chạy model lớn hơn và nhanh hơn.

In [ ]:
#@title Bước 1. Cấu hình
#@markdown **Cách gọi model**: `ollama` = chạy model mã nguồn mở trên GPU của Colab (khuyến nghị, không giới hạn số lượt gọi);
#@markdown `openai` = gọi API tương thích OpenAI (Groq, Gemini, OpenRouter...), bị giới hạn lượt gọi mỗi ngày.
BACKEND = "ollama"  #@param ["ollama", "openai"]

#@markdown ---
#@markdown **Model chạy trên Colab** (khi BACKEND = ollama). T4 (16 GB): `qwen3.5:9b`, `qwen3:8b`, `qwen3:14b`.
#@markdown L4/A100 (Colab Pro): thêm `qwen3.5:27b`, `gpt-oss:20b`. Có thể gõ tên model khác trên https://ollama.com/library
MODEL = "qwen3.5:9b"  #@param ["qwen3.5:9b", "qwen3:8b", "qwen3:14b", "qwen3.5:27b", "gpt-oss:20b"] {allow-input: true}

#@markdown ---
#@markdown **API** (khi BACKEND = openai). API key cất trong **Colab Secrets** (biểu tượng chìa khóa bên trái).
API_PRESET = "groq"  #@param ["groq", "gemini", "openrouter", "khac"]
API_MODEL = "openai/gpt-oss-120b"  #@param {type:"string"}
API_KEY_SECRET = "GROQ_API_KEY"  #@param {type:"string"}
API_BASE_URL_KHAC = ""  #@param {type:"string"}
API_RPM = 25  #@param {type:"number"}

#@markdown ---
#@markdown **Số câu**: 0 = đủ 500 câu như bài báo; số khác = lấy mẫu phân tầng theo CSDL × độ khó (vd 100 để chạy thử nhanh).
SAMPLE = 0  #@param {type:"integer"}
#@markdown Số token tối đa mỗi câu trả lời (bài báo: 800; model suy luận như gpt-oss nên đặt 2000–4000).
NUM_PREDICT = 800  #@param {type:"integer"}
#@markdown Lưu kết quả vào Google Drive để chạy tiếp khi Colab bị ngắt.
LUU_DRIVE = True  #@param {type:"boolean"}

import re
PRESETS = {"groq": "https://api.groq.com/openai/v1",
           "gemini": "https://generativelanguage.googleapis.com/v1beta/openai",
           "openrouter": "https://openrouter.ai/api/v1"}
API_BASE_URL = PRESETS.get(API_PRESET, API_BASE_URL_KHAC)
TEN_MODEL = MODEL if BACKEND == "ollama" else API_MODEL
RUN_NAME = re.sub(r"[^A-Za-z0-9._-]+", "_", f"{BACKEND}_{TEN_MODEL}") + (f"_mau{SAMPLE}" if SAMPLE else "_500cau")
REPO = "/content/vsqlplus"
DATA = "/content/data/minidev/MINIDEV"
OUT = f"/content/ketqua/{RUN_NAME}"
DRIVE_DIR = f"/content/drive/MyDrive/vsqlplus_ketqua/{RUN_NAME}"
# 5 cấu hình cần chạy LLM: base, vsql (tái hiện) và 3 nguồn bổ sung cho bỏ phiếu (CT7)
METHODS = ["base", "vsql", "auto_det", "hybrid", "hybrid_ret"]
print("Lần chạy:", RUN_NAME)
print("Kết quả :", OUT, "| Drive:", DRIVE_DIR if LUU_DRIVE else "(không lưu)")

In [ ]:
#@title Bước 2. Kiểm tra GPU và kết nối Google Drive
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "KHÔNG CÓ GPU")
if BACKEND == "ollama" and not gpu.stdout.strip():
    raise SystemExit("Chưa bật GPU: Runtime -> Change runtime type -> T4 GPU, rồi chạy lại từ Bước 1.")
if BACKEND == "ollama" and gpu.stdout.strip():
    vram = int(re.search(r"(\d+) MiB", gpu.stdout).group(1)) / 1024
    can = {"qwen3.5:27b": 20, "gpt-oss:20b": 16, "qwen3:14b": 12}.get(MODEL, 8)
    print(f"VRAM ~{vram:.0f} GB; {MODEL} cần khoảng {can} GB" + (" -> CẢNH BÁO: có thể không đủ, chọn model nhỏ hơn" if vram < can else " -> đủ"))
if LUU_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
#@title Bước 3. Tải mã nguồn và chạy kiểm thử (khoảng 1 phút)
import os
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q --depth 1 https://github.com/chloenguyen167/vsqlplus.git {REPO}
%cd {REPO}
!python tests/test_all.py | tail -1
!python tests/test_repair.py | tail -1

In [ ]:
#@title Bước 4. Tải BIRD mini-dev (~800 MB, khoảng 2–5 phút; lưu bản zip vào Drive để lần sau không phải tải lại)
import os, shutil
ZIP_DRIVE = "/content/drive/MyDrive/vsqlplus_ketqua/minidev.zip"
if not os.path.exists(f"{DATA}/mini_dev_sqlite.json"):
    os.makedirs("/content/data", exist_ok=True)
    if LUU_DRIVE and os.path.exists(ZIP_DRIVE):
        shutil.copy(ZIP_DRIVE, "/content/data/minidev.zip")
    else:
        !wget -q --show-progress -O /content/data/minidev.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip
        if LUU_DRIVE:
            os.makedirs(os.path.dirname(ZIP_DRIVE), exist_ok=True)
            shutil.copy("/content/data/minidev.zip", ZIP_DRIVE)
    # chỉ giải nén phần SQLite (bỏ bản MySQL/PostgreSQL)
    !unzip -q -o /content/data/minidev.zip "minidev/MINIDEV/*" -d /content/data
import json
print(len(json.load(open(f"{DATA}/mini_dev_sqlite.json"))), "câu;", len(os.listdir(f"{DATA}/dev_databases")), "CSDL")

## Bước 5. Khởi động model
Chạy **ô 5a** nếu `BACKEND = ollama`, hoặc **ô 5b** nếu `BACKEND = openai`. Ô không phù hợp sẽ tự bỏ qua.

In [ ]:
#@title Bước 5a. Cài Ollama, tải model và khởi động (khoảng 3–8 phút)
import subprocess, time, urllib.request, json, os, re
if BACKEND == "ollama":
    !apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -2 /tmp/ollama_install.log
    env = dict(os.environ, OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0",
               OLLAMA_NUM_PARALLEL="1", OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434")
    ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"),
                                   stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2); break
        except Exception:
            time.sleep(1)
    print("Ollama:", urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())
    !ollama pull {MODEL} 2>&1 | tail -1
    # thử một câu để đo tốc độ (lần đầu chậm vì nạp model vào GPU)
    body = {"model": MODEL, "stream": False, "think": False, "options": {"temperature": 0, "num_ctx": 16384, "num_predict": 60},
            "messages": [{"role": "user", "content": "Write one SQLite query that counts rows of table t. Return only SQL."}]}
    for lan in (1, 2):
        t = time.time()
        r = json.loads(urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:11434/api/chat",
            json.dumps(body).encode(), {"Content-Type": "application/json"}), timeout=600).read())
        print(f"Lần {lan}: {time.time()-t:.1f}s | {r['eval_count']/(r['eval_duration']/1e9):.0f} token/giây | trả lời: {r['message']['content'][:80]!r}")
    tl = r["message"]["content"]
    if "<think>" in tl or len(tl) > 200 or not re.search(r"select", tl, re.I):
        print("CẢNH BÁO: model vẫn 'suy nghĩ' dù đã tắt chế độ suy luận, mỗi câu sẽ rất chậm và dễ bị cắt. "
              "Hãy chọn model khác (vd qwen3.5:9b) hoặc tăng NUM_PREDICT lên 2000-4000.")
    !nvidia-smi --query-gpu=memory.used,memory.total --format=csv
else:
    print("Bỏ qua (BACKEND = openai).")

In [ ]:
#@title Bước 5b. Nạp API key từ Colab Secrets và thử gọi API
import os, json, urllib.request
if BACKEND == "openai":
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get(API_KEY_SECRET)
    hdr = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}", "Content-Type": "application/json"}
    try:   # liệt kê model để kiểm tra tên API_MODEL
        ds = json.loads(urllib.request.urlopen(urllib.request.Request(API_BASE_URL + "/models", headers=hdr), timeout=30).read())
        ten = sorted(m["id"] for m in ds.get("data", []))
        print(len(ten), "model, ví dụ:", ten[:40])
        if API_MODEL not in ten:
            print(f"CẢNH BÁO: không thấy '{API_MODEL}' trong danh sách, hãy sửa API_MODEL ở Bước 1.")
    except Exception as e:
        print("Không liệt kê được model:", e)
    body = {"model": API_MODEL, "max_tokens": NUM_PREDICT, "temperature": 0,
            "messages": [{"role": "user", "content": "Write one SQLite query that counts rows of table t. Return only SQL."}]}
    r = json.loads(urllib.request.urlopen(urllib.request.Request(API_BASE_URL + "/chat/completions",
                   json.dumps(body).encode(), hdr), timeout=120).read())
    print("Trả lời:", r["choices"][0]["message"]["content"][:200])
else:
    print("Bỏ qua (BACKEND = ollama).")

In [ ]:
#@title Bước 6. Đồng bộ kết quả với Google Drive (khôi phục phần đã chạy + tự sao lưu mỗi 5 phút)
import os, shutil, sqlite3, threading, time
os.makedirs(OUT, exist_ok=True)

def sao_luu(src=OUT, dst=DRIVE_DIR):
    """Chép thư mục kết quả sang Drive; file SQLite (cache LLM) chép bằng backup API để không bị hỏng khi đang ghi."""
    if not LUU_DRIVE:
        return
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        os.makedirs(os.path.join(dst, rel), exist_ok=True)
        for f in files:
            a, b = os.path.join(root, f), os.path.join(dst, rel, f)
            if f.endswith(".sqlite"):
                with sqlite3.connect(a) as s, sqlite3.connect(b + ".tmp") as d:
                    s.backup(d)
                os.replace(b + ".tmp", b)
            elif not os.path.exists(b) or os.path.getmtime(a) > os.path.getmtime(b) + 1:
                shutil.copy2(a, b)

if LUU_DRIVE and os.path.isdir(DRIVE_DIR):          # khôi phục lần chạy trước
    shutil.copytree(DRIVE_DIR, OUT, dirs_exist_ok=True)
    print("Đã khôi phục từ Drive:", sorted(os.listdir(OUT))[:12])

def _vong_sao_luu():
    while True:
        time.sleep(300)
        try:
            sao_luu()
        except Exception as e:
            print("[sao lưu] lỗi:", e)

if LUU_DRIVE and not globals().get("_sao_luu_dang_chay"):
    threading.Thread(target=_vong_sao_luu, daemon=True).start()
    _sao_luu_dang_chay = True
    print("Đang tự sao lưu", OUT, "->", DRIVE_DIR, "mỗi 5 phút")

In [ ]:
#@title Tham số dòng lệnh chung (chạy ô này sau Bước 6)
CHUNG = f"--data {DATA}/mini_dev_sqlite.json --db-root {DATA}/dev_databases"
if BACKEND == "ollama":
    LLM_ARGS = f"--model {MODEL} --num-predict {NUM_PREDICT}"
else:
    LLM_ARGS = (f"--backend openai --model {API_MODEL} --base-url {API_BASE_URL} --api-key-env OPENAI_API_KEY "
                f"--rpm {API_RPM} --num-predict {NUM_PREDICT}")
MAU = f"--sample {SAMPLE}" if SAMPLE else ""
print(CHUNG, LLM_ARGS, MAU)

In [ ]:
#@title Bước 7. Sinh view và kiểm tra số dòng (không gọi LLM, khoảng 1–2 phút)
!python run_experiment.py {CHUNG} --out {OUT} --views-only 2>&1 | grep -v "^\[schema\]"

In [ ]:
#@title Bước 8. Chạy thử 5 câu với cả 5 cấu hình (kiểm tra mọi thứ hoạt động, khoảng 2–5 phút)
!python run_experiment.py {CHUNG} {LLM_ARGS} --out /content/thu5cau --limit 5 --methods {",".join(METHODS)} 2>&1 | grep -E "Tong|base|vsql|auto_det|hybrid|LOI|Error|Trace" | head -20

## Bước 9. Chạy chính
Ô này chạy LLM cho 5 cấu hình (`base`, `vsql` và 3 nguồn bổ sung cho bỏ phiếu). Chương trình gom các prompt giống nhau phần
đầu để Ollama dùng lại bộ nhớ đệm, nên dòng `[prefetch ...] con lai ~N phut` cho biết thời gian còn lại.

**Nếu Colab bị ngắt:** mở lại notebook, chạy lại Bước 1 → 7, rồi chạy lại ô này. Phần đã chạy được khôi phục từ Drive
và được bỏ qua.

In [ ]:
#@title Bước 9. Chạy chính (lâu nhất: khoảng 2–4 giờ cho 500 câu với model 8–9B trên T4)
import subprocess, sys
cmd = f"python -u run_experiment.py {CHUNG} {LLM_ARGS} --out {OUT} --methods {','.join(METHODS)} {MAU}"
print(cmd)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    if not line.startswith(("[schema]", "    {")):
        print(line, end="")
p.wait()
sao_luu()
print("Xong, mã thoát:", p.returncode)

In [ ]:
#@title Bước 10. Áp dụng 2 cải tiến: CT5 sửa lỗi theo lược đồ, CT7 bỏ phiếu (không gọi LLM, khoảng 10–30 phút)
import subprocess
# CT5 cho từng cấu hình, chạy song song (mỗi cấu hình một tiến trình)
procs = [subprocess.Popen(f"python improve.py {OUT} {CHUNG} --fix --methods {m}", shell=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) for m in METHODS]
for p in procs:
    out, _ = p.communicate()
    print([l for l in out.splitlines() if l.startswith("[") or "Error" in l][-1:])
# CT7: bỏ phiếu giữa 4 cách biểu diễn lược đồ (đã qua CT5)
!python improve.py {OUT} {CHUNG} --vote mrv=base_fix,auto_det_fix,hybrid_ret_fix,hybrid_fix 2>&1 | grep -E "^\[|Error"
sao_luu()

In [ ]:
#@title Bước 11. Bảng kết quả, kiểm định thống kê và biểu đồ
import csv, sys
from IPython.display import Markdown, display
sys.path.insert(0, REPO)
from vsql import report
report.build(OUT)
CHON = ["base", "vsql", "base_fix", "vsql_fix", "mrv"]
TEN = {"base": "base (một giai đoạn)", "vsql": "V-SQL (tái hiện bài báo)", "base_fix": "base + CT5 sửa lỗi",
       "vsql_fix": "V-SQL + CT5 sửa lỗi", "mrv": "CT7 bỏ phiếu (kèm CT5)"}
S = {r["method"]: r for r in csv.DictReader(open(f"{OUT}/summary.csv"))}
bang = ["| Cấu hình | n | EX (%) | CI 95% | Lỗi sai tên bảng/cột (%) | Lượt gọi LLM/câu |", "|---|---|---|---|---|---|"]
for m in CHON:
    if m in S:
        x = S[m]
        bang.append(f"| {TEN[m]} | {x['n']} | **{float(x['ex']):.1f}** | {float(x['ci_lo']):.1f}–{float(x['ci_hi']):.1f} | "
                    f"{float(x['schema_hallucination']):.1f} | {float(x['calls']):.2f} |")
display(Markdown("### Kết quả chính\n" + "\n".join(bang)))
print("Kiểm định ghép cặp (McNemar, hiệu chỉnh Holm):")
print("  P1: base+CT5 vs base | P2: V-SQL+CT5 vs V-SQL | P5: CT7 vs base | P6: CT7 vs V-SQL")
!python hypotheses.py {OUT}

import matplotlib.pyplot as plt
m_co = [m for m in CHON if m in S]
ex = [float(S[m]["ex"]) for m in m_co]
lo = [float(S[m]["ex"]) - float(S[m]["ci_lo"]) for m in m_co]
hi = [float(S[m]["ci_hi"]) - float(S[m]["ex"]) for m in m_co]
mau = ["#1baf7a", "#1baf7a", "#2a78d6", "#2a78d6", "#2a78d6"][:len(m_co)]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh([TEN[m] for m in m_co][::-1], ex[::-1], xerr=[lo[::-1], hi[::-1]], color=mau[::-1], height=0.55, capsize=3)
for i, v in enumerate(ex[::-1]):
    ax.text(v + hi[::-1][i] + 0.5, i, f"{v:.1f}", va="center")
ax.set_xlabel("EX (%) và khoảng tin cậy 95%"); ax.set_title(f"{TEN_MODEL} – {S['base']['n']} câu BIRD mini-dev", loc="left")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.savefig(f"{OUT}/bieu_do_ex.png", dpi=160); plt.show()
display(Markdown(f"Bảng đầy đủ (mọi cấu hình, phân loại lỗi, chi phí): `{OUT}/summary.md`"))
sao_luu()

In [ ]:
#@title Bước 12. Tải kết quả về máy (file zip)
import shutil
from google.colab import files
z = shutil.make_archive(f"/content/{RUN_NAME}", "zip", OUT)
print("Kích thước:", round(os.path.getsize(z) / 1e6, 1), "MB")
files.download(z)

## Gặp lỗi?
- **`CUDA out of memory` / Ollama chạy rất chậm**: model quá lớn cho GPU. Chọn model nhỏ hơn ở Bước 1 (`qwen3.5:9b` hoặc `qwen3:8b` cho T4).
- **Colab ngắt giữa chừng**: chạy lại Bước 1 → 7 rồi Bước 9; phần đã xong được khôi phục từ Drive.
- **API báo 429 nhiều lần / hết hạn mức ngày**: giảm `API_RPM`, giảm `SAMPLE`, hoặc chạy tiếp vào ngày hôm sau (cache giữ các câu đã gọi).
- **Model trả lời rỗng hoặc bị cắt (model suy luận)**: tăng `NUM_PREDICT` lên 2000–4000.

Chi tiết: `docs/HUONG_DAN_COLAB.md`.